# Chemotactic recruitment

_Investigation `chemotactic-recruitment` — coder reproduction notebook._

**Question.** Does a secreted chemotactic cue drive directional migration and recruitment of
responder cells into the source region, and does inhibiting the chemotactic
response reduce that recruitment?

The reference vertical slice for the **Biological Claim Layer** — the proposal
that a biological CLAIM (not an execution model) is the canonical source, with
simulators as compile targets. Here a secreted chemotactic cue recruits responder
cells, and recruitment requires BOTH the cue and a competent response. The claim,
its mechanism, and its viva-cpm realization are authored as a backend-neutral
bundle (investigations/chemotactic-recruitment/claim-bundle/). The claim is now
carried by TWO realizations: the original PHENOMENOLOGICAL CPM realization (three
studies, chemotaxis lambda a stand-in for the response) and a finer-grained
RECEPTOR-LEVEL realization (two studies) in which the response is EMERGENT from a
per-cell Hill receptor-occupancy model calibrated to a cited chemokine-receptor
affinity. The semantic gap the first realization recorded is now closed by the
second, not hidden. See the "Claim layer" figure and How-to-read below.

---

This notebook re-runs each study with the workspace's own process-bigraph protocol and renders its figures. The text states the **question and parameters** only — the figures produced by each run are the results. Set `RERUN = False` in the setup cell to render the committed `runs.db` without re-simulating.


In [ ]:
"""Self-contained reproduction of this investigation.

Generated by vivarium-dashboard (notebook_export). Each study below is re-run
live with the workspace's own process-bigraph protocol and its figures are
rendered from the resulting runs.db.
"""
import os
import sys
from pathlib import Path

# Resolve the repository root robustly so this notebook runs from a fresh clone
# at ANY path with no setup (no env var, no path editing). Priority:
#   1. $VIVARIUM_REPO, if it points at a real directory;
#   2. walk up from the notebook's working directory for the repo markers
#      (a directory holding both 'workspace/' and 'pyproject.toml') — Jupyter
#      starts in the notebook's dir, so a committed notebook finds its own root;
#   3. the absolute path it was generated for (back-compat for old layouts);
#   4. the current working directory (last resort).
def _find_repo_root(_start):
    for _cand in (_start, *_start.parents):
        if (_cand / "workspace").is_dir() and (_cand / "pyproject.toml").is_file():
            return _cand
    return None

REPO = None
_env = os.environ.get("VIVARIUM_REPO")
if _env and Path(_env).is_dir():
    REPO = Path(_env)
if REPO is None:
    REPO = _find_repo_root(Path.cwd().resolve())
if REPO is None and Path('/home/runner/work/viva-cpm/viva-cpm').is_dir():
    REPO = Path('/home/runner/work/viva-cpm/viva-cpm')
if REPO is None:
    REPO = Path.cwd()
sys.path.insert(0, str(REPO))
# Composite specs use repo-root-relative paths (datasets, caches), and the
# workspace's runner/renderer assume cwd == repo root — so run from there.
os.chdir(REPO)

# Re-simulate from scratch? Set False to render the committed runs.db (fast).
RERUN = True

# --- standard process-bigraph protocol: register the workspace's Core ---
from pbg_cpm_studies.core import build_core
core = build_core()

# --- imported from the repo this notebook was generated for ---

from IPython.display import HTML, display

import contextlib as _contextlib, io as _io
@_contextlib.contextmanager
def quiet():
    """Silence the simulator's verbose per-step stdout so the notebook
    output stays readable (the figures below are the results)."""
    with _contextlib.redirect_stdout(_io.StringIO()):
        yield

import html as _htmlmod
def show_viz(_h, height=560):
    """Display a visualization's HTML in an isolated iframe.

    The figures embed their own scripts (e.g. Plotly); JupyterLab does not
    execute <script> tags from display(HTML(...)), so an iframe srcdoc is
    used instead — the browser runs the scripts inside the frame."""
    display(HTML(
        '<iframe srcdoc="{}" style="width:100%;height:{}px;border:0">'
        '</iframe>'.format(_htmlmod.escape(_h, quote=True), height)
    ))

import json as _json
def describe_spec(spec):
    """Print a composite spec's structure (parameters, processes, wiring)
    then the full editable dict. The spec is plain data — assign to any
    field (e.g. spec['state'][proc]['config'][...]) before building."""
    print("composite:", spec.get("name"))
    if spec.get("description"):
        print("description:", str(spec["description"]).strip())
    _params = spec.get("parameters") or {}
    if _params:
        print("\nparameters (filled into ${name} placeholders):")
        for _p, _pdef in _params.items():
            print(f"  {_p}: default={_pdef.get('default')!r}  type={_pdef.get('type')}")
    print("\nprocesses (node -> address):")
    for _node, _body in (spec.get("state") or {}).items():
        if not (isinstance(_body, dict) and _body.get("_type") == "process"):
            continue
        print(f"  {_node}  ->  {_body.get('address')}   interval={_body.get('interval')!r}")
        for _port in ("inputs", "outputs"):
            if _body.get(_port):
                print(f"      {_port} ports: {_body[_port]}")
    print("\nfull editable spec dict:")
    print(_json.dumps(spec, indent=2, default=str))

import base64 as _b64, pathlib as _pl
def _render_one(address, config, runs_db, study_yaml):
    """Generic figure renderer (no workspace render_study_viz.py):
    resolve an ``image:<relpath>`` visualization to displayable HTML,
    relative to the study directory."""
    addr = str(address or '')
    for _scheme in ('image:', 'file:', 'gif:', 'png:', 'svg:', 'jpg:', 'jpeg:'):
        if addr.startswith(_scheme):
            addr = addr[len(_scheme):]; break
    _p = _pl.Path(addr)
    if not _p.is_absolute():
        _p = _pl.Path(study_yaml).resolve().parent / _p
    if not _p.is_file():
        return f'<p style="color:#b91c1c">figure not found: {address}</p>'
    _suffix = _p.suffix.lower()
    if _suffix == '.svg':
        return _p.read_text(encoding='utf-8', errors='replace')
    if _suffix in ('.png', '.jpg', '.jpeg', '.gif', '.webp'):
        _mime = 'jpeg' if _suffix in ('.jpg', '.jpeg') else _suffix[1:]
        _data = _b64.b64encode(_p.read_bytes()).decode('ascii')
        return f'<img src="data:image/{_mime};base64,{_data}" style="max-width:100%"/>'
    if _suffix in ('.html', '.htm'):
        return _p.read_text(encoding='utf-8', errors='replace')
    return f'<p style="color:#6b7280">unsupported figure type: {address}</p>'

## Study: Baseline: cue + response recruits (`recruitment-baseline`)

**Question.** When source cells secrete a diffusible cue and responder cells carry a
competent chemotactic response, do the responders directionally migrate up the
gradient and accumulate at the source?

**Objective.** Simulate the CPM recruitment world (source slab + six responders in the gradient
zone), equilibrate the secreted field, and measure the recruitment index —
fraction of responders within 15 px of the source — over 500 MC sweeps.

**Hypothesis.** With the cue secreted (rate 10) and the response intact (lambda 14), responder
centres of mass move toward the source region and the recruitment index rises
well above zero.

**Claim.** Responders climb the secreted gradient and are recruited to the source: the
recruitment index rises from 0 to a plateau of ~0.67-0.83 (mean approach 0.61).


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.chemotaxis.recruitment` | 0 | cue_rate=10.0, chemo_lambda=14.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis.recruitment`** — `spec_pbg_cpm_studies_composites_chemotaxis_recruitment` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis.recruitment` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-baseline ===
STUDY = 'recruitment-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Claim layer — claim → mechanism → realization**


In [ ]:
# Claim layer — claim → mechanism → realization
show_viz(_render_one('local:ClaimLayerAtlas', {}, RUNS_DB, STUDY_YAML))

**Claim bundle — source (claim / mechanism / realization)**


In [ ]:
# Claim bundle — source (claim / mechanism / realization)
show_viz(_render_one('local:ClaimBundle', {}, RUNS_DB, STUDY_YAML))

**Recruitment over time (all conditions)**


In [ ]:
# Recruitment over time (all conditions)
show_viz(_render_one('local:ChemotaxisRecruitment', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Responders are recruited to the source | kind=recruitment_index condition=recruitment-baseline stat=final | op gt value 0.5 |


## Study: Intervention: block the response (`recruitment-inhibited`)

**Question.** If the chemotactic response is blocked (lambda -> 0) while the cue is still
secreted, is recruitment abolished?

**Objective.** Repeat the baseline world with the responder chemotaxis strength set to zero (the
intervention) and measure the recruitment index over 500 MC sweeps.

**Hypothesis.** With the cue present but the response inhibited, responders no longer climb the
gradient; the recruitment index stays near zero.

**Claim.** Blocking the chemotactic response abolishes recruitment: the recruitment index
stays at 0 (mean approach ~0), despite the cue being present.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `inhibited` | `pbg_cpm_studies.composites.chemotaxis.recruitment` | 0 | cue_rate=10.0, chemo_lambda=0.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis.recruitment`** — `spec_pbg_cpm_studies_composites_chemotaxis_recruitment` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis.recruitment` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-inhibited ===
STUDY = 'recruitment-inhibited'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Recruitment over time (all conditions)**


In [ ]:
# Recruitment over time (all conditions)
show_viz(_render_one('local:ChemotaxisRecruitment', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Blocking the response abolishes recruitment | kind=recruitment_index condition=recruitment-inhibited stat=max | op lt value 0.1 |


## Study: Control: no cue, no recruitment (`recruitment-adversarial`)

**Question.** If competent responders are placed with NO cue secreted, does the metric
correctly report no recruitment (rather than a spurious signal)?

**Objective.** Run the world with competent responders but the source secretion turned off, and
confirm the recruitment index and metric correctly report no recruitment.

**Hypothesis.** With the response intact (lambda 14) but the cue removed (rate 0), there is no
gradient to climb; the recruitment index stays at zero.

**Claim.** Without a cue, competent responders are not recruited: the recruitment index
stays at 0. The metric does not produce a false-positive recruitment signal.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `adversarial` | `pbg_cpm_studies.composites.chemotaxis.recruitment` | 0 | cue_rate=0.0, chemo_lambda=14.0 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis.recruitment`** — `spec_pbg_cpm_studies_composites_chemotaxis_recruitment` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis.recruitment` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-adversarial ===
STUDY = 'recruitment-adversarial'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Recruitment over time (all conditions)**


In [ ]:
# Recruitment over time (all conditions)
show_viz(_render_one('local:ChemotaxisRecruitment', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| No cue yields no recruitment (rejection) | kind=recruitment_index condition=recruitment-adversarial stat=max | op lt value 0.1 |


## Study: Receptor realization: cue + receptor-gated response recruits (`recruitment-receptor-baseline`)

**Question.** Does a receptor-level realization of chemotactic recruitment -- where each
responder activates from naive to chemotaxing only once its per-cell
receptor-occupancy Hill relation crosses threshold -- reproduce the
baseline recruitment claim, using a cited receptor Kd instead of a
hand-tuned chemotaxis strength?

**Objective.** Simulate the receptor-recruitment world (the recruitment_receptor
composite): naive(2)/activated(3) responder sub-types, each carrying a
ReceptorSubcell that computes Hill occupancy from the local cue field, and
measure the recruitment index over 500 MC sweeps across 5 seeds.

**Hypothesis.** With the cue secreted (rate 10) and the receptor-gated response intact
(chemo_lambda 14, kd 2.9 nM), responders whose Hill-occupancy crosses the
activation threshold chemotax up the gradient, and the recruitment index
rises well above zero, consistent with the phenomenological baseline.

**Claim.** Cue + receptor-gated response recruits: across 5 seeds the final
recruitment index is 0.5667 (95% CI 0.3451-0.7882), above the 0.5 pass
threshold, with responder activation gated by a receptor Kd cited to
Nasser 2009 rather than a hand-tuned chemotaxis strength.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `baseline` | `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor` | 0 | cue_rate=10.0, chemo_lambda=14.0, kd=2.9 |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor`** — `spec_pbg_cpm_studies_composites_chemotaxis_receptor_recruitment_receptor` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-receptor-baseline ===
STUDY = 'recruitment-receptor-baseline'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Recruitment over time (all conditions)**


In [ ]:
# Recruitment over time (all conditions)
show_viz(_render_one('local:ChemotaxisRecruitment', {}, RUNS_DB, STUDY_YAML))

**Recruitment scene — responders climb the chemokine gradient (animated, baseline)**


In [ ]:
# Recruitment scene — responders climb the chemokine gradient (animated, baseline)
show_viz(_render_one('local:ReceptorRecruitmentScene', {}, RUNS_DB, STUDY_YAML))

**Receptor occupancy law with the cited Kd (evidence → mechanism)**


In [ ]:
# Receptor occupancy law with the cited Kd (evidence → mechanism)
show_viz(_render_one('local:ReceptorOccupancyLaw', {}, RUNS_DB, STUDY_YAML))

**Receptor-gated recruitment (baseline vs blocked, CI ribbons)**


In [ ]:
# Receptor-gated recruitment (baseline vs blocked, CI ribbons)
show_viz(_render_one('local:ReceptorRecruitmentCurve', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Responders are recruited to the source (receptor realization) | kind=recruitment_index condition=recruitment-receptor-baseline stat=final | op gt value 0.5 |


## Study: Receptor realization: blocking the response abolishes recruitment (`recruitment-receptor-blocked`)

**Question.** In the receptor-level realization, if the activated response is forced off
(blocked=True, zeroing the activated-type chemotaxis lambda even for cells
that cross the receptor-occupancy activation threshold) while the cue is
still secreted, is recruitment abolished?

**Objective.** Repeat the receptor-recruitment world with blocked=True (the intervention)
and measure the recruitment index over 500 MC sweeps across 5 seeds.

**Hypothesis.** With the cue present (rate 10) but the receptor-gated response disabled
(blocked=True), responders may still activate (naive -> activated) but no
longer chemotax, so the recruitment index stays near zero.

**Claim.** Blocking the response downstream of receptor activation abolishes
recruitment: the final recruitment index is 0.0667 (95% CI 0.0-0.1467),
below the 0.1 pass threshold, despite the cue being present and receptors
still able to bind ligand.


### Parameters

| simulation | composite | steps | params |
| --- | --- | --- | --- |
| `blocked` | `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor` | 0 | cue_rate=10.0, chemo_lambda=14.0, kd=2.9, blocked=True |


### Specification (process-bigraph) — load, inspect, edit

Each composite is a process-bigraph *document*: named processes (`_type: process`) bound to an `address`, wired by `inputs`/`outputs` ports over shared stores. For every composite below the first cell loads the spec into a plain **editable Python dict** and prints its structure; the second cell is a **control panel** listing every configuration value and per-process `interval` so you can tweak any of them. Your edits are read when the composite is built and run, in the **Run** section.


**Composite `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor`** — `spec_pbg_cpm_studies_composites_chemotaxis_receptor_recruitment_receptor` (a plain, editable dict)


_composite spec file for `pbg_cpm_studies.composites.chemotaxis_receptor.recruitment_receptor` not found under `pbg_cpm_studies/composites/` — skipped._


### Run

_Set the runtime (`STEPS`) and step size (`INTERVAL`), then run. Each simulation builds the (edited) spec above and writes `runs.db`; the figures below read it. Set `RERUN = False` to skip re-simulating._


In [ ]:
# === Study: recruitment-receptor-blocked ===
STUDY = 'recruitment-receptor-blocked'
STUDY_DIR = REPO / 'workspace/studies' / STUDY
STUDY_YAML = str(STUDY_DIR / "study.yaml")
RUNS_DB = str(STUDY_DIR / "runs.db")

print("No recorded runs for this study; nothing to reproduce.")

### Visualizations

_Results are shown by the figures below, produced by the run above._


**Recruitment over time (all conditions)**


In [ ]:
# Recruitment over time (all conditions)
show_viz(_render_one('local:ChemotaxisRecruitment', {}, RUNS_DB, STUDY_YAML))

**Recruitment scene — responders climb the chemokine gradient (animated, baseline)**


In [ ]:
# Recruitment scene — responders climb the chemokine gradient (animated, baseline)
show_viz(_render_one('local:ReceptorRecruitmentScene', {}, RUNS_DB, STUDY_YAML))

**Receptor occupancy law with the cited Kd (evidence → mechanism)**


In [ ]:
# Receptor occupancy law with the cited Kd (evidence → mechanism)
show_viz(_render_one('local:ReceptorOccupancyLaw', {}, RUNS_DB, STUDY_YAML))

**Receptor-gated recruitment (baseline vs blocked, CI ribbons)**


In [ ]:
# Receptor-gated recruitment (baseline vs blocked, CI ribbons)
show_viz(_render_one('local:ReceptorRecruitmentCurve', {}, RUNS_DB, STUDY_YAML))

### Acceptance criteria

_Pre-registered checks (criteria/thresholds only — run the cells above to evaluate them)._

| test | measures | passes if |
| --- | --- | --- |
| Blocking the response abolishes recruitment (receptor realization) | kind=recruitment_index condition=recruitment-receptor-blocked stat=final | op lt value 0.1 |


## Open decisions
- Should the Biological Claim Layer compiler now regenerate this study set from a hand-authored claim bundle?
